#  Plant Leaf Disease Classification — End-to-End

Transfer learning on the **New Plant Diseases Dataset** (~87,900 images, 38 crop-disease classes) to classify a leaf photo into its crop + disease, with **Grad-CAM** explainability and a **held-out test** evaluation.

**Runtime:** set *Runtime → Change runtime type → GPU* before running.

This notebook uses the reusable code in the project's `src/` package (so the notebook stays readable and the logic isn't duplicated). Pipeline:

1. Setup & GPU check
2. Download the dataset (`kagglehub`)
3. Quick EDA — class counts + sample leaves
4. Build `tf.data` pipelines (train / val / **held-out test**)
5. Train — Phase 1 (frozen backbone) → Phase 2 (fine-tuning)
6. Training curves
7. Evaluate on the held-out test set — confusion matrix + per-class F1
8. **Grad-CAM** — see what the model looks at
9. Save the model & predict on unseen demo images


## 1 · Setup

Clone the repo to get the `src/` package, then install dependencies. **Edit `REPO_URL`** to point at your fork.

In [ ]:
import os

# 👉 EDIT THIS to your repository URL after you push the project to GitHub.
REPO_URL = "https://github.com/YOUR_USERNAME/plant-disease-classifier.git"

# If the src/ package isn't already here (i.e. we're on a fresh Colab VM), clone it.
if not os.path.exists("src"):
    !git clone -q $REPO_URL
    %cd plant-disease-classifier

!pip install -q -r requirements.txt
print("Setup complete.")

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", bool(gpus), gpus)
if not gpus:
    print("⚠️  No GPU detected — training will be slow. Enable it via Runtime → Change runtime type → GPU.")

## 2 · Download the dataset

`kagglehub` downloads and caches the dataset. On Colab/Kaggle no API token is needed for this public dataset. The helper below also transparently handles the dataset's awkward *doubly-nested* folder layout.

In [ ]:
from src.data import make_datasets

# Downloads (first run only) and builds train / val / held-out test pipelines.
train_ds, val_ds, test_ds, class_names = make_datasets()
print(f"\n{len(class_names)} classes")
print(class_names[:5], "...")

## 3 · Quick EDA

How many images per class, and what do the leaves look like?

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from src.data import locate_data
from src.utils import prettify_label

train_dir, _ = locate_data()  # cached path, no re-download
counts = {c: len(list((train_dir / c).glob("*"))) for c in class_names}

plt.figure(figsize=(14, 8))
order = sorted(counts, key=counts.get)
plt.barh([prettify_label(c) for c in order], [counts[c] for c in order], color="#16a34a")
plt.title(f"Training images per class (total = {sum(counts.values()):,})")
plt.xlabel("images"); plt.tight_layout(); plt.show()

print("min/class:", min(counts.values()), " max/class:", max(counts.values()))

In [ ]:
# Sample leaves — one batch from the training set.
imgs, labels = next(iter(train_ds))
plt.figure(figsize=(13, 10))
for i in range(min(12, imgs.shape[0])):
    plt.subplot(3, 4, i + 1)
    plt.imshow(imgs[i].numpy().astype("uint8"))
    plt.title(prettify_label(class_names[int(labels[i])]), fontsize=8)
    plt.axis("off")
plt.tight_layout(); plt.show()

## 4 · The model

MobileNetV2 pretrained on ImageNet + a fresh classification head. Augmentation and the backbone's `preprocess_input` are **inside the model graph**, so the saved model takes raw pixels and can't suffer train/serve preprocessing drift.

Swap `backbone="resnet50"` or `"efficientnetb0"` to benchmark alternatives, or `"cnn"` for the from-scratch baseline.

In [ ]:
from src.models import build_transfer_model, compile_model
from src.config import CFG

model = build_transfer_model(backbone="mobilenetv2", num_classes=len(class_names))
compile_model(model, lr=CFG.head_lr)
model.summary()

## 5 · Training

**Phase 1 — feature extraction:** the backbone is frozen; only the new head learns.
**Phase 2 — fine-tuning:** the top ~30% of backbone layers are unfrozen and trained with a tiny learning rate (BatchNorm stays frozen).

`EarlyStopping` restores the best weights, so it's safe to leave the epoch counts generous.

In [ ]:
import time
from src.train import _callbacks

histories = []

print("=== Phase 1: training the classification head ===")
t0 = time.time()
h1 = model.fit(train_ds, validation_data=val_ds,
               epochs=CFG.head_epochs, callbacks=_callbacks())
histories.append(h1.history)
print(f"Phase 1 done in {(time.time()-t0)/60:.1f} min")

In [ ]:
from src.models import unfreeze_for_finetuning

n = unfreeze_for_finetuning(model)
print(f"=== Phase 2: fine-tuning {n} backbone layers ===")
compile_model(model, lr=CFG.fine_tune_lr)   # recompile after changing trainability

t0 = time.time()
h2 = model.fit(train_ds, validation_data=val_ds,
               epochs=CFG.fine_tune_epochs, callbacks=_callbacks())
histories.append(h2.history)
print(f"Phase 2 done in {(time.time()-t0)/60:.1f} min")

## 6 · Training curves

In [ ]:
from src.utils import plot_history
from IPython.display import Image, display

CFG.ensure_dirs()
path = plot_history(histories, CFG.assets_dir / "accuracy_loss_graph.png")
display(Image(str(path)))

## 7 · Save the model

Persist the model + the exact class-name ordering so inference and the app can reload them.

In [ ]:
from src.utils import save_json

model.save(CFG.model_path)
save_json(class_names, CFG.class_names_path)
print("Saved:", CFG.model_path)
print("Saved:", CFG.class_names_path)

## 8 · Evaluate on the held-out test set

This split was **never** used for training or early stopping, so these numbers are an honest estimate of generalisation. Produces overall + per-class metrics and the confusion matrix.

In [ ]:
from src.evaluate import evaluate

metrics = evaluate(model=model, test_ds=test_ds, class_names=class_names)

In [ ]:
display(Image(str(CFG.assets_dir / "confusion_matrix.png")))

In [ ]:
# The hardest confusions — usually visually-similar diseases on the same crop.
import pandas as pd
pd.DataFrame(metrics["most_confused"]).head(10)

## 9 · Grad-CAM — explainability

For a black-box CNN to be trustworthy, it should focus on the actual lesions. Grad-CAM overlays the model's attention on the leaf. Green/blue = ignored, red/yellow = influential.

In [ ]:
from src.gradcam import make_gradcam_heatmap, overlay_heatmap

# Grab a few test images and visualise prediction + attention.
imgs, labels = next(iter(test_ds))
plt.figure(figsize=(14, 9))
for i in range(6):
    raw = imgs[i].numpy().astype("uint8")
    batch = raw.astype("float32")[None, ...]
    probs = model.predict(batch, verbose=0)[0]
    pred = int(probs.argmax())
    heat, _ = make_gradcam_heatmap(batch, model, pred_index=pred)
    cam = overlay_heatmap(raw, heat, alpha=0.45)

    plt.subplot(2, 3, i + 1)
    plt.imshow(cam)
    ok = "✅" if pred == int(labels[i]) else "❌"
    plt.title(f"{ok} {prettify_label(class_names[pred])}\n{probs[pred]*100:.1f}%", fontsize=8)
    plt.axis("off")
plt.suptitle("Grad-CAM: where the model looks", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 10 · Predict on the loose demo images

The dataset ships 33 unlabelled `test/` images whose class is encoded in the filename — a nice sanity check on data the model has never seen in any split.

In [ ]:
from src.data import download_dataset
from src.predict import predict

root = download_dataset()
demo = [p for p in root.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        and "test" in str(p).lower()][:6]

if demo:
    plt.figure(figsize=(14, 9))
    for i, img_path in enumerate(demo):
        r = predict(img_path, model=model, class_names=class_names)
        plt.subplot(2, 3, i + 1)
        plt.imshow(r["image"])
        plt.title(f"{r['prediction']}\n{r['confidence']*100:.1f}%  |  file: {img_path.name}",
                  fontsize=7)
        plt.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No loose demo images found in this dataset version.")

---
### ✅ Done

You now have a trained model (`models/plant_disease_model.keras`), evaluation artefacts, and Grad-CAM visualisations.

**Next:** download `models/` locally and run the demo app with `streamlit run app/app.py`, or deploy it (Dockerfile + `app/requirements.txt` included). Paste your test metrics into the README results table.
